# DDPG on LunarLanderContinuous-v2

**Deep Deterministic Policy Gradient (DDPG)** extends the DPG algorithm with deep neural networks. It is an off-policy, model-free algorithm for **continuous action spaces** that uses:

- A **deterministic actor** that outputs the action directly.
- A **critic** that estimates $Q(s, a)$.
- **Target networks** for both actor and critic (soft Polyak updates).
- **Exploration noise** added to the deterministic policy during rollout collection.

> Paper: [Continuous control with deep reinforcement learning](https://arxiv.org/abs/1509.02971)

In this notebook we train DDPG on **LunarLanderContinuous-v2** using `lit-sb3`.

> **Note**: DDPG is a special case of TD3 with a single critic and no target-policy smoothing.


## Setup


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import gymnasium as gym
import numpy as np

from lit_sb3 import RLEnvDataModule, RLTrainer
from lit_sb3.algorithms.ddpg.ddpg import DDPG
from lit_sb3.core.noise import NormalActionNoise

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir ./

## Action noise

Because DDPG uses a deterministic policy, we add **Normal action noise** during training to encourage exploration. The noise has the same dimensionality as the action space.


In [ ]:
env_name = "LunarLanderContinuous-v3"

In [ ]:
env = gym.make(env_name)
n_actions = env.action_space.shape[0]
action_noise = NormalActionNoise(
    mean=np.zeros(n_actions),
    sigma=0.1 * np.ones(n_actions),
)
env.close()

## Create the model

Key hyperparameters:

| Parameter         | Description                                              |
| ----------------- | -------------------------------------------------------- |
| `action_noise`    | Exploration noise injected into the deterministic policy |
| `tau`             | Polyak coefficient for soft target-network updates       |
| `learning_starts` | Warm-up steps with random actions before learning        |


In [ ]:
model = DDPG(
    "MlpPolicy",
    verbose=1,
    learning_rate=1e-3,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    gradient_steps=1,
    action_noise=action_noise,
    policy_kwargs={"net_arch": [256, 256]},
)
print(model.policy_class.__name__)

## Train

Train through the explicit `RLTrainer` entry point.


In [ ]:
datamodule = RLEnvDataModule(train_env=env_name)
trainer = RLTrainer(max_steps=50_000)
trainer.fit(model, datamodule=datamodule)
print(model.policy)

## Evaluate


In [ ]:
eval_results = trainer.test(
    env=gym.make(env_name),
    n_eval_episodes=10,
)
eval_results[0]